# 03 — Evaluation Comparison

Aggregates `evaluation_report.json` files produced across all cron runs and
generates the thesis comparison table and figures.

**Prerequisite**: at least one complete run of `scripts/collect_and_detect.sh`
so that `ml/output/run_*/evaluation_report.json` exists.

## 1. Setup

In [ ]:
import json
import pathlib

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ML_DIR     = pathlib.Path("..").resolve()
OUTPUT_DIR = ML_DIR / "output"
FIGURES_DIR = OUTPUT_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

DETECTOR_ORDER = [
    "metrics_isolation_forest",
    "logs_isolation_forest",
    "rule_based",
    "combined_OR",
    "combined_AND",
]
DETECTOR_LABELS = {
    "metrics_isolation_forest": "Metrics IF",
    "logs_isolation_forest":    "Logs IF",
    "rule_based":               "Rule-based",
    "combined_OR":              "Combined OR",
    "combined_AND":             "Combined AND",
}

plt.rcParams.update({"figure.dpi": 110})
%matplotlib inline

## 2. Load All Evaluation Reports

In [ ]:
report_paths = sorted(OUTPUT_DIR.glob("run_*/evaluation_report.json"))
print(f"Found {len(report_paths)} report(s):")
for p in report_paths:
    print(f"  {p.parent.name}")

if not report_paths:
    raise FileNotFoundError(
        f"No run_*/evaluation_report.json found in {OUTPUT_DIR}. "
        "Run collect_and_detect.sh first."
    )

In [ ]:
clf_records = []    # classification metrics per run × detector
ttd_records = []    # time-to-detection per run × detector × incident

for rp in report_paths:
    run_ts = rp.parent.name.replace("run_", "")
    with open(rp) as f:
        report = json.load(f)

    detectors = report.get("detectors", {})
    for det, data in detectors.items():
        clf = data.get("classification", {})
        clf_records.append({
            "run_ts":    run_ts,
            "detector":  det,
            "precision": clf.get("precision"),
            "recall":    clf.get("recall"),
            "f1":        clf.get("f1"),
            "tp":        clf.get("tp"),
            "fp":        clf.get("fp"),
            "fn":        clf.get("fn"),
            "tn":        clf.get("tn"),
        })
        for inc_id, ttd_s in data.get("time_to_detection", {}).items():
            inc_meta = next(
                (i for i in report.get("incidents", []) if i.get("id") == inc_id),
                {}
            )
            ttd_records.append({
                "run_ts":   run_ts,
                "detector": det,
                "incident": inc_id,
                "type":     inc_meta.get("type", "unknown"),
                "ttd_s":    ttd_s,
            })

df_clf = pd.DataFrame(clf_records)
df_ttd = pd.DataFrame(ttd_records)
print(f"Classification rows: {len(df_clf)}")
print(f"TTD rows           : {len(df_ttd)}")
df_clf.head()

## 3. Metrics Summary Table

In [ ]:
# Mean across all runs per detector
summary = (
    df_clf
    .groupby("detector")[["precision", "recall", "f1", "tp", "fp", "fn"]]
    .mean()
    .reindex([d for d in DETECTOR_ORDER if d in df_clf["detector"].unique()])
    .rename(index=DETECTOR_LABELS)
    .round(3)
)

display(
    summary.style
    .highlight_max(subset=["precision", "recall", "f1"], color="#c6efce")
    .highlight_min(subset=["fp", "fn"],                  color="#c6efce")
    .format("{:.3f}", subset=["precision", "recall", "f1"])
    .set_caption("Mean classification metrics across all runs")
)

## 4. Detection Performance Charts

In [ ]:
present = [d for d in DETECTOR_ORDER if d in df_clf["detector"].unique()]
means   = df_clf.groupby("detector")[["precision", "recall", "f1"]].mean().reindex(present)
labels  = [DETECTOR_LABELS[d] for d in present]

x  = np.arange(len(present))
w  = 0.25

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 bar chart
ax = axes[0]
ax.bar(x, means["f1"], color="steelblue", edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha="right")
ax.set_ylim(0, 1.05)
ax.set_title("F1 Score per Detector")
ax.set_ylabel("F1")

# Precision vs Recall grouped bar
ax = axes[1]
ax.bar(x - w/2, means["precision"], w, label="Precision", color="steelblue", edgecolor="white")
ax.bar(x + w/2, means["recall"],    w, label="Recall",    color="tomato",    edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha="right")
ax.set_ylim(0, 1.05)
ax.set_title("Precision vs Recall")
ax.legend()

plt.tight_layout()
fig.savefig(FIGURES_DIR / "detection_performance.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved → {FIGURES_DIR / 'detection_performance.png'}")

## 5. Time-to-Detection Analysis

In [ ]:
if df_ttd.empty:
    print("No TTD data available yet.")
else:
    df_ttd["ttd_min"] = df_ttd["ttd_s"] / 60

    inc_types = df_ttd["type"].unique().tolist()
    n_types   = len(inc_types)
    fig, axes = plt.subplots(1, max(n_types, 1), figsize=(7 * max(n_types, 1), 5), squeeze=False)

    for ax, inc_type in zip(axes[0], inc_types):
        subset = df_ttd[df_ttd["type"] == inc_type]
        present_d = [d for d in DETECTOR_ORDER if d in subset["detector"].unique()]
        data_by_det = [subset[subset["detector"] == d]["ttd_min"].dropna().values for d in present_d]
        ax.boxplot(data_by_det, labels=[DETECTOR_LABELS[d] for d in present_d])
        ax.set_title(f"TTD — incident type: {inc_type}")
        ax.set_ylabel("Time to detection (min)")
        ax.tick_params(axis="x", rotation=20)

    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "time_to_detection.png", dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved → {FIGURES_DIR / 'time_to_detection.png'}")

    print("\nMean TTD (minutes) per detector × incident type:")
    display(
        df_ttd.groupby(["detector", "type"])["ttd_min"]
        .mean()
        .unstack("type")
        .reindex([d for d in DETECTOR_ORDER if d in df_ttd["detector"].unique()])
        .rename(index=DETECTOR_LABELS)
        .round(2)
    )

## 6. Time-Series Overlay (single representative run)

In [ ]:
# Pick the most recent run that has both anomaly files.
run_dirs = sorted(OUTPUT_DIR.glob("run_*"), reverse=True)
selected_run = None
for rd in run_dirs:
    if (rd / "metrics_anomalies.csv").exists() and (rd / "logs_anomalies.csv").exists():
        selected_run = rd
        break

if selected_run is None:
    print("No complete run found with both anomaly CSVs.")
else:
    print(f"Using run: {selected_run.name}")
    m_anom = pd.read_csv(selected_run / "metrics_anomalies.csv", index_col=0, parse_dates=True)
    l_anom = pd.read_csv(selected_run / "logs_anomalies.csv",    index_col=0, parse_dates=True)

    # Load incidents for overlay
    with open(ML_DIR / "data_ingest" / "incidents.json") as f:
        raw_inc = json.load(f)
    incidents = raw_inc if isinstance(raw_inc, list) else raw_inc.get("incidents", [])

    def shade_incidents(ax, incidents):
        colors = {"errors": "tomato", "slow": "orange"}
        for inc in incidents:
            s = pd.Timestamp(inc["start"], tz="UTC")
            e = pd.Timestamp(inc["end"],   tz="UTC")
            ax.axvspan(s, e, color=colors.get(inc["type"], "gray"), alpha=0.15)

    fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

    # Panel 1: Metrics anomaly score
    ax = axes[0]
    ax.plot(m_anom.index, m_anom["anomaly_score"], color="steelblue", lw=0.9, label="anomaly_score")
    ax.scatter(m_anom.index[m_anom["is_anomaly"] == 1],
               m_anom["anomaly_score"][m_anom["is_anomaly"] == 1],
               color="red", s=18, zorder=5, label="flagged")
    shade_incidents(ax, incidents)
    ax.set_title("Metrics IsolationForest"); ax.set_ylabel("score"); ax.legend(fontsize=8)

    # Panel 2: Logs anomaly score
    ax = axes[1]
    if "anomaly_score" in l_anom.columns:
        ax.plot(l_anom.index, l_anom["anomaly_score"], color="purple", lw=0.9, label="anomaly_score")
        ax.scatter(l_anom.index[l_anom["is_anomaly"] == 1],
                   l_anom["anomaly_score"][l_anom["is_anomaly"] == 1],
                   color="red", s=18, zorder=5, label="flagged")
    shade_incidents(ax, incidents)
    ax.set_title("Logs IsolationForest"); ax.set_ylabel("score"); ax.legend(fontsize=8)

    # Panel 3: Combined OR
    ax = axes[2]
    # Align on a common 1-min index
    combined_idx = m_anom.index.union(l_anom.index)
    m_ser = m_anom["is_anomaly"].reindex(combined_idx).fillna(0)
    l_ser = l_anom["is_anomaly"].reindex(combined_idx).fillna(0) if "is_anomaly" in l_anom.columns else pd.Series(0, index=combined_idx)
    combined = ((m_ser + l_ser) >= 1).astype(int)
    ax.fill_between(combined.index, combined, color="tomato", alpha=0.6, label="Combined OR")
    shade_incidents(ax, incidents)
    ax.set_title("Combined OR (any detector fires)"); ax.set_ylabel("anomaly"); ax.legend(fontsize=8)

    legend_handles = [
        mpatches.Patch(color="tomato", alpha=0.3, label="incident: errors"),
        mpatches.Patch(color="orange", alpha=0.3, label="incident: slow"),
    ]
    axes[0].legend(handles=axes[0].get_legend_handles_labels()[0] + legend_handles, fontsize=8)

    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "timeseries_overlay.png", dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved → {FIGURES_DIR / 'timeseries_overlay.png'}")

## 7. Combined Detector Analysis

In [ ]:
# Compare OR vs AND: precision/recall trade-off
or_and = df_clf[df_clf["detector"].isin(["combined_OR", "combined_AND"])]

if or_and.empty:
    print("No combined detector data in reports yet.")
else:
    fig, ax = plt.subplots(figsize=(7, 4))
    for det, grp in or_and.groupby("detector"):
        ax.scatter(grp["recall"], grp["precision"],
                   label=DETECTOR_LABELS.get(det, det), s=60, zorder=5)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("OR vs AND: precision / recall trade-off (each point = one run)")
    ax.set_xlim(0, 1.05); ax.set_ylim(0, 1.05)
    ax.legend()
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "or_vs_and.png", dpi=300, bbox_inches="tight")
    plt.show()

    print("\nFalse positive reduction (OR → AND):")
    for run in or_and["run_ts"].unique():
        sub = or_and[or_and["run_ts"] == run].set_index("detector")
        if "combined_OR" in sub.index and "combined_AND" in sub.index:
            fp_or  = sub.loc["combined_OR",  "fp"]
            fp_and = sub.loc["combined_AND", "fp"]
            print(f"  {run}: FP OR={fp_or:.0f}  AND={fp_and:.0f}  "
                  f"(reduction {100*(fp_or-fp_and)/max(fp_or,1):.0f}%)")

## 8. Thesis Figures Export

In [ ]:
# All figures are already saved at 300 DPI in FIGURES_DIR by the cells above.
# This cell lists what was exported.
print(f"Thesis figures in: {FIGURES_DIR}")
for p in sorted(FIGURES_DIR.glob("*.png")):
    print(f"  {p.name}")

In [ ]:
# Export the summary table as LaTeX for the thesis
if not summary.empty:
    latex = summary[["precision", "recall", "f1"]].to_latex(
        float_format="{:.3f}".format,
        caption="Anomaly detector comparison (mean over all experiment runs)",
        label="tab:detector_comparison",
    )
    tex_path = FIGURES_DIR / "detector_comparison.tex"
    tex_path.write_text(latex)
    print(f"LaTeX table saved → {tex_path}")
    print(latex)